In [5]:
# Паттерн Стратегия через классы

from abc import ABC, abstractmethod
from collections.abc import Sequence
from decimal import Decimal
from typing import NamedTuple, Optional

class Customer(NamedTuple):
    name: str
    fidelity: int

class LineItem(NamedTuple):
    product: str
    quantity: int
    price: Decimal
    def total(self) -> Decimal:
        return self.price * self.quantity
    
class Order(NamedTuple): # контекст
    customer: Customer
    cart: Sequence[LineItem]
    promotion: Optional['Promotion'] = None
    def total(self) -> Decimal:
        totals = (item.total() for item in self.cart)
        return sum(totals, start=Decimal(0))
    def due(self) -> Decimal:
        if self.promotion is None:
            discount = Decimal(0)
        else:
            discount = self.promotion.discount(self)
        return self.total() - discount
    def __repr__(self):
        return f'<Order total: {self.total():.2f} due: {self.due():.2f}>'
    
class Promotion(ABC): # Стратегия: абстрактный базовый класс
    @abstractmethod
    def discount(self, order: Order) -> Decimal:
        '''Вернуть скидку в виде положительной суммы в долларах'''

class FidelityPromo(Promotion): # первая конкретная стратегия
    '''5%-ная скидка для заказчиков, имеющих не менее 1000 баллов лояльности'''
    def discount(self, order: Order) -> Decimal:
        rate = Decimal('0.05')
        if order.customer.fidelity >= 1000:
            return order.total() * rate
        return Decimal(0)
    
class BulkItemPromo(Promotion): # вторая конкретная стратегия
    '''10%-ная скидка для каждой позиции LineItem, в которой заказано не менее 20 единиц'''
    def discount(self, order: Order) -> Decimal:
        discount = Decimal(0)
        for item in order.cart:
            if item.quantity >= 20:
                discount += item.total() * Decimal('0.1')
                return discount
            
class LargeOrderPromo(Promotion): # третья конкретная стратегия
    '''7%-ная скидка для заказов, включающих не менее 10 различных позиций'''
    def discount(self, order: Order) -> Decimal:
        distinct_items = {item.product for item in order.cart}
        if len(distinct_items) >= 10:
            return order.total() * Decimal('0.07')
        return Decimal(0)


In [7]:
# Паттерн Страегия через функции

from collections.abc import Sequence
from dataclasses import dataclass
from decimal import Decimal
from typing import Optional, Callable, NamedTuple
class Customer(NamedTuple):
    name: str
    fidelity: int
class LineItem(NamedTuple):
    product: str
    quantity: int
    price: Decimal
    def total(self):
        return self.price * self.quantity
    
@dataclass(frozen=True)
class Order: # контекст
    customer: Customer
    cart: Sequence[LineItem]
    promotion: Optional[Callable[['Order'], Decimal]] = None
    def total(self) -> Decimal:
        totals = (item.total() for item in self.cart)
        return sum(totals, start=Decimal(0))
    def due(self) -> Decimal:
        if self.promotion is None:
            discount = Decimal(0)
        else:
            discount = self.promotion(self)
        return self.total() - discount
    def __repr__(self):
        return f'<Order total: {self.total():.2f} due: {self.due():.2f}>'

def fidelity_promo(order: Order) -> Decimal:
    """5%-ная скидка для заказчиков, имеющих не менее 1000 баллов лояльности"""
    if order.customer.fidelity >= 1000:
        return order.total() * Decimal('0.05')
    return Decimal(0)

def bulk_item_promo(order: Order) -> Decimal:
    """10%-ная скидка для каждой позиции LineItem, в которой заказано
       не менее 20 единиц"""
    discount = Decimal(0)
    for item in order.cart:
        if item.quantity >= 20:
            discount += item.total() * Decimal('0.1')
        return discount
    
def large_order_promo(order: Order) -> Decimal:
    """7%-ная скидка для заказов, включающих не менее 10 различных позиций"""
    distinct_items = {item.product for item in order.cart}
    if len(distinct_items) >= 10:
        return order.total() * Decimal('0.07')
    return Decimal(0)

promos = [fidelity_promo, bulk_item_promo, large_order_promo]
def best_promo(order: Order) -> Decimal:
   """Вычислить наибольшую скидку"""
   return max(promo(order) for promo in promos) 